# 01 · Ingest and validate retail sales
**Question:** Which categories drive sales growth, and does that growth preserve gross margin?

Our fictional retailer supplies one row per order, with one category per order. All money
is USD; dates are calendar dates. This dataset covers January–June 2025 and is synthetic.
Returns are outside this example's contract. A few bad rows and exact duplicates are deliberate.

Run all three notebooks with `uv run run-pipeline`, or run this notebook top to bottom and
copy its printed `run_id` into the parameters cell of notebooks 02 and 03.
Start Jupyter with `uv run jupyter lab` from the project root and choose its Python kernel.

One setup call installs automatic per-cell logging. Optional `# step:` comments give log
entries meaningful names. Errors keep their normal Jupyter traceback and fail this run.

In [ ]:
# The runner injects these values. For manual use, copy run_id from notebook 01.
run_id = None
input_csv = None
demo_failure = None

In [ ]:
from notebook_pipeline import start_notebook

session = start_notebook("01_ingest", run_id=run_id)
run_id = session.run_id
from pathlib import Path

import pandas as pd

from notebook_pipeline.errors import DataContractError
from notebook_pipeline.quality import (
    enforce_quality,
    read_sales,
    valid_sales_rows,
    validate_columns,
)
from notebook_pipeline.runtime import write_json

## Read the CSV
Missing/unreadable input is fatal. The default dataset is bundled locally.

In [ ]:
# step: read_csv
source = Path(input_csv) if input_csv else session.run.root / "data/raw/sales.csv"
if demo_failure == "missing-file":
    source = session.run.path / "deliberately_missing.csv"
sales = read_sales(source)
sales.head()

In [ ]:
# step: validate_schema
if demo_failure == "schema":
    sales = sales.drop(columns="unit_price")
validate_columns(sales)
raw_rows = len(sales)
sales.shape

## Normalize and enforce quality
Exact duplicate records can be removed safely. Conflicting records with the same order ID
are fatal. Invalid rows can be excluded only if they are at most **2%** of nonduplicate rows;
this is an explicit example business policy. No financial values are imputed.

In [ ]:
# step: normalize_types
sales = sales.copy()
for column in ["order_id", "region", "category"]:
    sales[column] = sales[column].astype("string").str.strip()
sales["region"] = sales["region"].str.title()
sales["category"] = sales["category"].str.title()
sales["order_date"] = pd.to_datetime(sales["order_date"], format="%Y-%m-%d", errors="coerce")
for column in ["quantity", "unit_price", "unit_cost"]:
    sales[column] = pd.to_numeric(sales[column], errors="coerce")

In [ ]:
# step: deduplicate
duplicate_rows = int(sales.duplicated().sum())
sales = sales.drop_duplicates().copy()
if sales["order_id"].dropna().duplicated().any():
    raise DataContractError("Conflicting records share an order_id; resolve them at the source.")
if duplicate_rows:
    session.log.warning("data.exact_duplicates_removed", extra={"fields": {"rows": duplicate_rows}})

In [ ]:
# step: enforce_row_quality
valid = valid_sales_rows(sales)
rejected_rows = int((~valid).sum())
enforce_quality(len(sales), rejected_rows)
clean = sales.loc[valid].copy()
clean["quantity"] = clean["quantity"].astype("int64")
if rejected_rows:
    session.log.warning("data.invalid_rows_excluded", extra={"fields": {"rows": rejected_rows}})
quality_report = {
    "raw_rows": raw_rows,
    "duplicates_removed": duplicate_rows,
    "invalid_rows_excluded": rejected_rows,
    "clean_rows": len(clean),
    "max_rejected_fraction": 0.02,
}
quality_report

In [ ]:
# step: publish_clean_data
with session.output("clean_sales.parquet") as path:
    clean.to_parquet(path, index=False)
with session.output("quality_report.json") as path:
    write_json(path, quality_report)

## Complete the handoff
Outputs become consumable only when this entire final cell succeeds. A failed stage cannot
be resumed in this example: fix the cause, then start a fresh run from notebook 01.
In interactive Jupyter, later cells can still be clicked after an error, but publishing and
downstream handoff are blocked. The automated runner stops immediately.

In [ ]:
# step: finish_ingestion
print(f"Use run_id = {run_id!r} in notebook 02.")
session.finish()